# Prophet R04: persistent pilot
Pinned source, GPU gate, audited corpus and vocabulary, whole-document validation.
The two arms and three seeds remain experiments, not an AGI claim.
Run the final release cell only after recording results.

In [ ]:
# Mount access explicitly approved by the user on 2026-09-19.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
persistent = Path('/content/drive/MyDrive/Prophet_AGI/R04')
persistent.mkdir(parents=True, exist_ok=True)


In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path
repo = Path('/content/Prophet_AGI')
revision = 'e5720d0b774977455b1920a6f67b5df078377f11'
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/speed25200-cyber/Prophet_AGI.git', str(repo)], check=True)
subprocess.run(['git', 'fetch', 'origin', 'claude/prophet-v03-memory-context'], cwd=repo, check=True)
subprocess.run(['git', 'checkout', revision], cwd=repo, check=True)
os.environ['TRITON_F32_DEFAULT'] = 'tf32x3'
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo)+'[dev,gpu]', 'datasets==5.0.1', 'huggingface-hub==1.32.0'], check=True)
with (persistent/'gpu-gate.log').open('w') as gate_log:
    gate = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_gpu.py', '-v', '--tb=short'], cwd=repo, stdout=gate_log, stderr=subprocess.STDOUT, timeout=900)
print((persistent/'gpu-gate.log').read_text())
gate.check_returncode()
corpus = repo / 'data/fineweb-pilot-v1'
cache = persistent / 'corpus-v1'
if not corpus.exists():
    if cache.exists():
        shutil.copytree(cache, corpus)
    else:
        subprocess.run([sys.executable, 'scripts/prepare_pilot.py', '--out', str(corpus)], cwd=repo, check=True)
        subprocess.run([sys.executable, 'scripts/train_tokenizer.py', '--data-root', str(corpus/'train'), '--out', str(corpus/'tokenizer.json'), '--vocab-size', '32768', '--max-docs', '10000'], cwd=repo, check=True)
        subprocess.run([sys.executable, 'scripts/run_r04_pilot.py', '--corpus', str(corpus), '--variant', 'loop', '--seed', '0', '--out', str(persistent/'loop-seed0'), '--dry-run'], cwd=repo, check=True)
        shutil.copytree(corpus, persistent/'corpus-v1.building')
        (persistent/'corpus-v1.building').rename(cache)
print('R04_SETUP_VERIFIED', revision, flush=True)


In [ ]:
# A bounded part of a fixed 4096-step experiment. Repeat to resume.
variant, seed = 'loop', 0
assert variant in ('loop', 'plain') and seed in (0, 1, 2)
assert 'r04_process' not in globals() or r04_process.poll() is not None
import time
log_path = Path('/content') / f'{variant}-seed{seed}-session-{time.time_ns()}.log'
persistent_log = persistent / log_path.name
log_stream = log_path.open('a', buffering=1)
r04_process = subprocess.Popen([sys.executable, '-u', 'scripts/run_r04_pilot.py', '--corpus', str(corpus), '--variant', variant, '--seed', str(seed), '--out', str(persistent/f'{variant}-seed{seed}'), '--max-session-steps', '128', '--session-minutes', '45'], cwd=repo, stdout=log_stream, stderr=subprocess.STDOUT, start_new_session=True)
log_stream.close()
print('R04_PROCESS', r04_process.pid, 'LOG', str(log_path), flush=True)
# For an intentional early stop, run r04_process.terminate(), then wait for EXIT 0.
# The signal requests a completed-step checkpoint; notebook interrupts are isolated.


In [ ]:
print('EXIT', r04_process.poll())
print(log_path.read_text()[-16000:])


In [ ]:
# Only release after the session finished, the checkpoint and evaluation exist,
# and their metadata has been retained in the experiment record.
assert r04_process.poll() == 0
import shutil
shutil.copyfile(log_path, persistent_log)
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()
